In [3]:
# %pip install -U "google-cloud-aiplatform[ray]" "ray[default]" google-cloud-bigquery pandas pyarrow db-dtypes statsforecast mlforecast utilsforecast lightgbm xgboost scikit-learn

## Configuration

In [ ]:
from pathlib import Path

PROJECT_ID = "your-project-id"
REGION = "us-central1"
STAGING_BUCKET = "gs://your-staging-bucket"
VERTEX_SERVICE_ACCOUNT = "your-vertex-sa@your-project-id.iam.gserviceaccount.com"
NETWORK = None  # e.g. "projects/PROJECT_NUMBER/global/networks/VPC_NAME" or None

# --- BQ source ---
BQ_DATASET = "your_dataset"
BQ_TABLE = "your_table"
UID_COL = "uid_col"
DATE_COL = "date_col"
TARGET_COL = "target_col"

SYS_ID_VALUE = "your_sys_id"
UID_DELIMITER = "_"

# --- Forecasting params ---
FREQ = "MS"
SEASON_LENGTH = 12
HORIZON = 12
VALIDATION_HORIZON = 12
MIN_HISTORY_FOR_FULL_AUDIT = 24
ML_LAGS = [1, 12]
ML_AUDIT_SHARDS = 4  # 0 = auto, else force shard count for distributed ML audit
MAX_SERIES = 100  # 0 to process all

# --- Ray cluster shape (4 worker nodes + 1 head = 80 vCPU on n1-standard-16) ---
HEAD_MACHINE_TYPE = "n1-standard-16"
WORKER_MACHINE_TYPE = "n1-standard-16"
WORKER_NODE_COUNT = 4
BOOT_DISK_SIZE_GB = 200
RAY_VERSION = "2.47"
PYTHON_VERSION = "3.11"
CLUSTER_NAME = f"nixtla-forecast-{SYS_ID_VALUE.lower().replace('_', '-')}"
REUSE_EXISTING_CLUSTER = True   # if a cluster with CLUSTER_NAME exists, reuse it instead of creating
DELETE_CLUSTER_AFTER_RUN = False # set True only if you want this notebook to tear it down

# --- BQ output ---
OUTPUT_DATASET = "Nixtla_final_forecast_results_custom_job"
FORECAST_TABLE = "nixtla_final_forecast_results"
METRICS_TABLE = "customer_model_performance_metrics"
CHAMPIONS_TABLE = "best_model_metadata"
RUN_LOGS_TABLE = "pipeline_run_logs"

ENTRYPOINT_DIR = Path("ray_job")
ENTRYPOINT_DIR.mkdir(exist_ok=True)
ENTRYPOINT_SCRIPT = ENTRYPOINT_DIR / "train_nixtla_ray_cluster.py"


## Create output BigQuery dataset (idempotent)

In [ ]:
from google.cloud import bigquery as _bq

_bq_client = _bq.Client(project=PROJECT_ID)
_dataset_ref = _bq.Dataset(f"{PROJECT_ID}.{OUTPUT_DATASET}")
_dataset_ref.location = REGION.split("-")[0].upper() if REGION.startswith("us") else REGION
_bq_client.create_dataset(_dataset_ref, exists_ok=True)
print(f"BQ dataset ready: {PROJECT_ID}.{OUTPUT_DATASET}")

## Write the Ray training script

Same logic as the Custom Job version, but `ray.init(address="auto")` so it connects to the cluster head when submitted as a Ray Job.

In [ ]:
training_script = r'''
import argparse
import json
import os
import re
import time
import ray

import numpy as np
import pandas as pd
from google.cloud import bigquery

from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA, AutoETS, AutoTheta, MSTL, ARCH, GARCH,
    ADIDA, CrostonOptimized, Naive, SeasonalNaive,
)
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--project_id", required=True)
    p.add_argument("--bq_dataset", required=True)
    p.add_argument("--bq_table", required=True)
    p.add_argument("--uid_col", required=True)
    p.add_argument("--date_col", required=True)
    p.add_argument("--target_col", required=True)
    p.add_argument("--sys_id_value", required=True)
    p.add_argument("--uid_delimiter", required=True)
    p.add_argument("--freq", default="MS")
    p.add_argument("--season_length", type=int, default=12)
    p.add_argument("--horizon", type=int, default=12)
    p.add_argument("--validation_horizon", type=int, default=12)
    p.add_argument("--min_history_for_full_audit", type=int, default=24)
    p.add_argument("--max_series", type=int, default=0)
    p.add_argument("--ml_lags", default="1,12")
    p.add_argument("--ml_audit_shards", type=int, default=0)
    p.add_argument("--output_dataset", required=True)
    p.add_argument("--forecast_table", required=True)
    p.add_argument("--metrics_table", required=True)
    p.add_argument("--champions_table", required=True)
    p.add_argument("--run_logs_table", required=True)
    return p.parse_args()


def utc_now() -> pd.Timestamp:
    return pd.Timestamp.now(tz="UTC").tz_localize(None)


def build_query(project_id, dataset, table):
    return f"""
    WITH base AS (
      SELECT
        CAST({{uid_col}} AS STRING) AS unique_id,
        DATE({{date_col}}) AS ds,
        CAST({{target_col}} AS FLOAT64) AS y
      FROM `{project_id}.{dataset}.{table}`
      WHERE {{uid_col}} IS NOT NULL AND {{date_col}} IS NOT NULL AND {{target_col}} IS NOT NULL
    ),
    parsed AS (
      SELECT SPLIT(unique_id, @uid_delimiter)[SAFE_OFFSET(0)] AS sys_id, unique_id, ds, y
      FROM base
    )
    SELECT unique_id, ds, y, sys_id
    FROM parsed WHERE sys_id = @sys_id_value
    ORDER BY unique_id, ds
    """


def read_data(args):
    client = bigquery.Client(project=args.project_id)
    query = build_query(args.project_id, args.bq_dataset, args.bq_table).format(
        uid_col=args.uid_col, date_col=args.date_col, target_col=args.target_col,
    )
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("uid_delimiter", "STRING", args.uid_delimiter),
        bigquery.ScalarQueryParameter("sys_id_value", "STRING", args.sys_id_value),
    ])
    df = client.query(query, job_config=job_config).to_dataframe(create_bqstorage_client=True)
    if df.empty:
        raise ValueError(f"No rows returned for sys_id={args.sys_id_value}")
    df["ds"] = pd.to_datetime(df["ds"])
    if args.max_series and args.max_series > 0:
        uids = df["unique_id"].unique()[:args.max_series]
        df = df[df["unique_id"].isin(uids)]
        print(f"[max_series] Limiting to {len(uids)} series for testing.")
    return df


def aggregate_monthly(df):
    df = df.copy()
    df["ds"] = df["ds"].dt.to_period("M").dt.to_timestamp()
    return (df.groupby(["unique_id", "ds"], as_index=False)["y"].sum()
              .sort_values(["unique_id", "ds"]).reset_index(drop=True))


def split_ids_by_history(df, min_history):
    counts = df.groupby("unique_id").size()
    return counts[counts >= min_history].index.tolist(), counts[counts < min_history].index.tolist()


def split_train_valid(df, validation_horizon):
    train_parts, valid_parts = [], []
    for _, grp in df.groupby("unique_id", sort=False):
        grp = grp.sort_values("ds").reset_index(drop=True)
        if len(grp) <= validation_horizon:
            continue
        train_parts.append(grp.iloc[:-validation_horizon].copy())
        valid_parts.append(grp.iloc[-validation_horizon:].copy())
    if not train_parts or not valid_parts:
        raise ValueError("Not enough history to build train/validation split.")
    return pd.concat(train_parts, ignore_index=True), pd.concat(valid_parts, ignore_index=True)


def build_stats_models(season_length):
    return [
        AutoARIMA(season_length=season_length),
        AutoETS(season_length=season_length),
        AutoTheta(season_length=season_length),
        MSTL(season_length=[season_length], trend_forecaster=AutoARIMA(season_length=1)),
        ARCH(), GARCH(), ADIDA(), CrostonOptimized(),
        Naive(), SeasonalNaive(season_length=season_length),
    ]


@ray.remote
def _stats_audit_uid_remote(uid_records, horizon, freq, season_length):
    try:
        df = pd.DataFrame(uid_records)
        df["ds"] = pd.to_datetime(df["ds"])
        df = df.sort_values("ds").reset_index(drop=True)
        sf = StatsForecast(models=build_stats_models(season_length), freq=freq, n_jobs=1)
        preds = sf.forecast(df=df, h=horizon)
        preds["ds"] = pd.to_datetime(preds["ds"])
        return preds.to_dict(orient="records")
    except Exception as e:
        uid = uid_records[0].get("unique_id") if uid_records else "<unknown>"
        print(f"[stats_audit_error] uid={uid} error={e!r}")
        return []


def run_stats_audit(train_df, horizon, freq, season_length):
    futures = [
        _stats_audit_uid_remote.remote(grp.to_dict(orient="records"), horizon, freq, season_length)
        for _, grp in train_df.groupby("unique_id", sort=False)
    ]
    results = ray.get(futures)
    rows = [row for batch in results for row in batch]
    if not rows:
        return pd.DataFrame(columns=["unique_id", "ds"])
    out = pd.DataFrame(rows)
    out["ds"] = pd.to_datetime(out["ds"])
    return out.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def build_mlforecast(lags, freq, num_threads=-1):
    models = {
        "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                                subsample=0.9, colsample_bytree=0.9, random_state=42,
                                objective="reg:squarederror", n_jobs=1),
        "LightGBM": LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=64,
                                  subsample=0.9, colsample_bytree=0.9, random_state=42,
                                  verbosity=-1, n_jobs=1),
    }
    lag_transforms = {
        lags[0]: [RollingMean(window_size=3), RollingMean(window_size=6), RollingMean(window_size=12)],
        lags[-1]: [RollingMean(window_size=3)],
    }
    return MLForecast(models=models, freq=freq, lags=lags, lag_transforms=lag_transforms,
                      date_features=["month", "quarter", "year"], num_threads=num_threads)


@ray.remote
def _ml_audit_shard_remote(shard_records, horizon, lags, freq):
    try:
        shard_df = pd.DataFrame(shard_records)
        shard_df["ds"] = pd.to_datetime(shard_df["ds"])
        shard_df = shard_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)
        fcst = build_mlforecast(lags, freq, num_threads=1)
        fcst.fit(shard_df, id_col="unique_id", time_col="ds", target_col="y", static_features=[])
        preds = fcst.predict(horizon)
        preds["ds"] = pd.to_datetime(preds["ds"])
        return preds.to_dict(orient="records")
    except Exception as e:
        print(f"[ml_audit_error] shard_failed error={e!r}")
        return []


def _balanced_uid_shards(train_df, n_shards):
    counts = train_df.groupby("unique_id").size().sort_values(ascending=False)
    shards = [[] for _ in range(max(1, n_shards))]
    loads = [0 for _ in range(max(1, n_shards))]
    for uid, cnt in counts.items():
        idx = int(np.argmin(loads))
        shards[idx].append(uid)
        loads[idx] += int(cnt)
    return [s for s in shards if s]


def run_ml_audit_distributed(train_df, horizon, lags, freq, n_shards):
    n_unique = int(train_df["unique_id"].nunique())
    n_shards = max(1, min(int(n_shards), n_unique))
    uid_shards = _balanced_uid_shards(train_df, n_shards)
    futures = []
    shard_row_counts = []
    for shard_uids in uid_shards:
        shard_df = train_df[train_df["unique_id"].isin(shard_uids)].copy()
        shard_row_counts.append(int(len(shard_df)))
        futures.append(
            _ml_audit_shard_remote.remote(
                shard_df.to_dict(orient="records"),
                horizon,
                lags,
                freq,
            )
        )

    print(f"[ml_audit] shard_row_counts={shard_row_counts}")
    results = ray.get(futures)
    rows = [row for batch in results for row in batch]
    if not rows:
        raise ValueError("Distributed ML audit returned no predictions.")
    out = pd.DataFrame(rows)
    out["ds"] = pd.to_datetime(out["ds"])
    return out.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def evaluate_predictions(valid_df, preds):
    merged = valid_df.merge(preds, on=["unique_id", "ds"], how="inner")
    model_cols = [c for c in merged.columns if c not in {"unique_id", "ds", "y"}]
    if not model_cols:
        raise ValueError("No forecast columns found for evaluation.")
    metric_df = evaluate(merged, metrics=[rmse], models=model_cols, id_col="unique_id", target_col="y")
    metric_df = metric_df[metric_df["metric"] == "rmse"].drop(columns=["metric"])
    return (metric_df.melt(id_vars=["unique_id"], var_name="model", value_name="rmse")
                     .sort_values(["unique_id", "rmse", "model"]).reset_index(drop=True))


def build_champions(stats_metrics, ml_metrics, fallback_ids, sys_id_value):
    all_metrics = pd.concat([stats_metrics, ml_metrics], ignore_index=True)
    best = (all_metrics.sort_values(["unique_id", "rmse", "model"])
                       .groupby("unique_id", as_index=False).first()
                       .rename(columns={"model": "best_model_name"}))
    best["sys_id"] = sys_id_value
    if fallback_ids:
        fb = pd.DataFrame({"unique_id": fallback_ids, "best_model_name": "Naive_Fallback",
                           "rmse": np.nan, "sys_id": sys_id_value})
        best = pd.concat([best, fb], ignore_index=True)
    return best[["sys_id", "unique_id", "best_model_name", "rmse"]].sort_values(["unique_id"]).reset_index(drop=True)


def normalize_model_name(model_name):
    if not isinstance(model_name, str):
        return model_name
    name = model_name.strip()
    if not name:
        return name

    # StatsForecast may emit parameterized labels (e.g. ARCH(1), GARCH(1,1)).
    compact = name.replace(" ", "")
    if compact.startswith("ARCH("):
        return "ARCH"
    if compact.startswith("GARCH("):
        return "GARCH"

    aliases = {
        "NaiveFallback": "Naive_Fallback",
    }
    return aliases.get(name, name)


def fit_single_stats_model(model_name, uid_df, horizon, freq, season_length):
    resolved_model_name = normalize_model_name(model_name)
    model_map = {
        "AutoARIMA": AutoARIMA(season_length=season_length),
        "AutoETS": AutoETS(season_length=season_length),
        "AutoTheta": AutoTheta(season_length=season_length),
        "MSTL": MSTL(season_length=[season_length], trend_forecaster=AutoARIMA(season_length=1)),
        "ARCH": ARCH(), "GARCH": GARCH(), "ADIDA": ADIDA(),
        "CrostonOptimized": CrostonOptimized(), "Naive": Naive(),
        "SeasonalNaive": SeasonalNaive(season_length=season_length),
        "Naive_Fallback": Naive(),
    }
    if resolved_model_name not in model_map:
        supported = sorted(model_map.keys())
        raise ValueError(f"Unsupported stats model: {model_name} (resolved={resolved_model_name}); supported={supported}")
    sf = StatsForecast(models=[model_map[resolved_model_name]], freq=freq, n_jobs=1)
    pred = sf.forecast(df=uid_df, h=horizon)
    pred["ds"] = pd.to_datetime(pred["ds"])
    forecast_col = [c for c in pred.columns if c not in {"unique_id", "ds"}][0]
    return pred.rename(columns={forecast_col: "Forecast"})[["unique_id", "ds", "Forecast"]]


def fit_single_ml_model(model_name, uid_df, horizon, lags, freq):
    if model_name == "XGBoost":
        models = {"fcst_temp": XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                                            subsample=0.9, colsample_bytree=0.9, random_state=42,
                                            objective="reg:squarederror", n_jobs=1)}
    elif model_name == "LightGBM":
        models = {"fcst_temp": LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=64,
                                             subsample=0.9, colsample_bytree=0.9, random_state=42,
                                             verbosity=-1, n_jobs=1)}
    else:
        raise ValueError(f"Unsupported ML model: {model_name}")
    fcst = MLForecast(models=models, freq=freq, lags=lags,
                      lag_transforms={lags[0]: [RollingMean(window_size=3), RollingMean(window_size=6), RollingMean(window_size=12)],
                                      lags[-1]: [RollingMean(window_size=3)]},
                      date_features=["month", "quarter", "year"], num_threads=1)
    fcst.fit(uid_df, id_col="unique_id", time_col="ds", target_col="y", static_features=[])
    pred = fcst.predict(horizon)
    pred["ds"] = pd.to_datetime(pred["ds"])
    return pred.rename(columns={"fcst_temp": "Forecast"})[["unique_id", "ds", "Forecast"]]


@ray.remote
def _forecast_uid_remote(uid, grp_records, best_model, horizon, freq, season_length, sys_id_value, lags):
    ml_models = {"XGBoost", "LightGBM"}
    try:
        grp = pd.DataFrame(grp_records)
        grp["ds"] = pd.to_datetime(grp["ds"])
        grp = grp.sort_values("ds").reset_index(drop=True)
        resolved_model = normalize_model_name(best_model)
        if resolved_model in ml_models:
            pred = fit_single_ml_model(resolved_model, grp, horizon, lags, freq)
        else:
            pred = fit_single_stats_model(resolved_model, grp, horizon, freq, season_length)
        pred["sys_id"] = sys_id_value
        pred["Best_model_name"] = best_model
        return pred[["sys_id", "unique_id", "ds", "Forecast", "Best_model_name"]].to_dict(orient="records")
    except Exception as e:
        print(f"[forecast_error] uid={uid} model={best_model} error={e!r}")
        return []


def refit_and_forecast(full_df, champions_df, horizon, freq, season_length, sys_id_value, lags):
    champ_map = dict(zip(champions_df["unique_id"], champions_df["best_model_name"]))
    futures = []
    for uid, grp in full_df.groupby("unique_id", sort=False):
        best_model = champ_map.get(uid)
        if best_model is None:
            continue
        futures.append(_forecast_uid_remote.remote(
            uid, grp.to_dict(orient="records"), best_model,
            horizon, freq, season_length, sys_id_value, lags))
    results = ray.get(futures)
    rows = [row for batch in results for row in batch]
    if not rows:
        return pd.DataFrame(columns=["sys_id", "unique_id", "ds", "Forecast", "Best_model_name", "created_at"])
    out = pd.DataFrame(rows)
    out["ds"] = pd.to_datetime(out["ds"])
    out["created_at"] = utc_now()
    return out.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def build_metrics_output(stats_metrics, ml_metrics, champions_df):
    all_metrics = pd.concat([stats_metrics, ml_metrics], ignore_index=True)
    if all_metrics.empty:
        return pd.DataFrame()
    wide = all_metrics.pivot_table(index="unique_id", columns="model", values="rmse", aggfunc="first")
    wide.columns = [re.sub(r"[^a-zA-Z0-9_]", "_", f"{col}_RMSE") for col in wide.columns]
    wide = wide.reset_index()
    best = champions_df[["unique_id", "best_model_name", "rmse"]].rename(
        columns={"best_model_name": "Best_model_name", "rmse": "Best_model_RMSE"})
    wide = wide.merge(best, on="unique_id", how="left")
    wide["created_at"] = utc_now()
    return wide


def write_bq(df, table_fqn, project_id, write_disposition="WRITE_APPEND"):
    client = bigquery.Client(project=project_id)
    df = df.drop(columns=["sys_id"], errors="ignore")
    job_config = bigquery.LoadJobConfig(write_disposition=write_disposition)
    if write_disposition == "WRITE_APPEND":
        job_config.schema_update_options = [bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION]
    client.load_table_from_dataframe(df, table_fqn, job_config=job_config).result()


def write_run_logs(df, table_fqn, project_id):
    client = bigquery.Client(project=project_id)
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
    job_config.schema_update_options = [bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION]
    client.load_table_from_dataframe(df, table_fqn, job_config=job_config).result()


def main():
    args = parse_args()
    ml_lags = [int(x) for x in args.ml_lags.split(",")]

    run_started_at = utc_now()

    # When run as a Ray Job on a Vertex Ray cluster, address="auto" finds the head.
    ray.init(address="auto", ignore_reinit_error=True, logging_level="WARNING")
    cluster_resources = ray.cluster_resources()
    print(f"[ray] cluster_resources={cluster_resources}")
    total_cpus = int(cluster_resources.get("CPU", 1))
    print(f"[ray] total_cpus_in_cluster={total_cpus}")

    t0 = time.perf_counter()
    raw_df = read_data(args)
    full_df = aggregate_monthly(raw_df)
    data_read_seconds = round(time.perf_counter() - t0, 3)

    eligible_ids, fallback_ids = split_ids_by_history(full_df, args.min_history_for_full_audit)
    eligible_df = full_df[full_df["unique_id"].isin(eligible_ids)].copy()

    stats_metrics = pd.DataFrame(columns=["unique_id", "model", "rmse"])
    ml_metrics = pd.DataFrame(columns=["unique_id", "model", "rmse"])
    ml_shards_used = 0
    stats_audit_seconds = 0.0
    ml_audit_seconds = 0.0

    if not eligible_df.empty:
        train_df, valid_df = split_train_valid(eligible_df, args.validation_horizon)

        # Stats audit: per-uid Ray fanout across the whole cluster.
        n_series = eligible_df["unique_id"].nunique()
        print(f"[audit] running stats audit via Ray ({n_series} series x 10 models, ~{total_cpus} parallel slots)")
        t_stats = time.perf_counter()
        stats_preds = run_stats_audit(train_df, args.validation_horizon, args.freq, args.season_length)
        stats_audit_seconds = round(time.perf_counter() - t_stats, 3)

        # ML audit: shard by unique_id and distribute across Ray workers.
        auto_shards = max(1, min(total_cpus // 2 if total_cpus > 1 else 1, n_series))
        requested_ml_shards = args.ml_audit_shards if args.ml_audit_shards and args.ml_audit_shards > 0 else auto_shards
        ml_shards_used = max(1, min(int(requested_ml_shards), int(n_series)))
        print(f"[audit] running distributed ml audit via Ray (requested_shards={requested_ml_shards}, used_shards={ml_shards_used})")
        t_ml = time.perf_counter()
        ml_preds = run_ml_audit_distributed(train_df, args.validation_horizon, ml_lags, args.freq, ml_shards_used)
        ml_audit_seconds = round(time.perf_counter() - t_ml, 3)

        stats_metrics = evaluate_predictions(valid_df, stats_preds)
        ml_metrics = evaluate_predictions(valid_df, ml_preds)

    champions_df = build_champions(stats_metrics, ml_metrics, fallback_ids, args.sys_id_value)
    metrics_df = build_metrics_output(stats_metrics, ml_metrics, champions_df)

    forecast_df = refit_and_forecast(
        full_df, champions_df, args.horizon, args.freq, args.season_length, args.sys_id_value, ml_lags)

    forecast_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.forecast_table}"
    metrics_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.metrics_table}"
    champions_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.champions_table}"
    run_logs_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.run_logs_table}"

    t_bq = time.perf_counter()
    if not metrics_df.empty:
        write_bq(metrics_df, metrics_table_fqn, args.project_id, write_disposition="WRITE_TRUNCATE")
    write_bq(champions_df, champions_table_fqn, args.project_id)
    write_bq(forecast_df, forecast_table_fqn, args.project_id)
    bq_write_seconds = round(time.perf_counter() - t_bq, 3)

    run_finished_at = utc_now()

    summary = {
        "sys_id": args.sys_id_value,
        "max_series": args.max_series,
        "series_count": int(full_df["unique_id"].nunique()),
        "eligible_ids_for_full_audit": len(eligible_ids),
        "fallback_ids": len(fallback_ids),
        "ml_audit_shards_used": int(ml_shards_used),
        "metrics_rows": int(len(metrics_df)),
        "champion_rows": int(len(champions_df)),
        "forecast_rows": int(len(forecast_df)),
        "data_read_seconds": data_read_seconds,
        "stats_audit_seconds": stats_audit_seconds,
        "ml_audit_seconds": ml_audit_seconds,
        "bq_write_seconds": bq_write_seconds,
        "run_started_at": run_started_at,
        "run_finished_at": run_finished_at,
        "run_duration_seconds": round((run_finished_at - run_started_at).total_seconds(), 3),
    }

    run_logs_df = pd.DataFrame([summary])
    run_logs_df["created_at"] = utc_now()
    write_run_logs(run_logs_df, run_logs_table_fqn, args.project_id)

    print(json.dumps(summary, indent=2, default=str))


if __name__ == "__main__":
    main()
'''

ENTRYPOINT_SCRIPT.write_text(training_script)
print(f"Wrote {ENTRYPOINT_SCRIPT.resolve()}")

## Initialise Vertex AI

In [ ]:
from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=STAGING_BUCKET,
)

## Create a Vertex Ray cluster

Provisions 1 head + 4 workers = 80 vCPU. Takes ~5–10 min. **You are billed for all VMs while the cluster is up — delete it at the end.**

In [ ]:
from google.cloud.aiplatform import vertex_ray
from google.cloud.aiplatform.vertex_ray import Resources

head_node = Resources(
    machine_type=HEAD_MACHINE_TYPE,
    node_count=1,
    boot_disk_size_gb=BOOT_DISK_SIZE_GB,
)

worker_nodes = [
    Resources(
        machine_type=WORKER_MACHINE_TYPE,
        node_count=WORKER_NODE_COUNT,
        boot_disk_size_gb=BOOT_DISK_SIZE_GB,
    ),
]


def _find_existing_cluster(name: str):
    for c in vertex_ray.list_ray_clusters():
        # cluster_resource_name looks like .../persistentResources/<name>
        if c.cluster_resource_name.rsplit("/", 1)[-1] == name:
            return c.cluster_resource_name
    return None


cluster_resource_name = None
if REUSE_EXISTING_CLUSTER:
    cluster_resource_name = _find_existing_cluster(CLUSTER_NAME)
    if cluster_resource_name:
        print(f"Reusing existing cluster: {cluster_resource_name}")

if cluster_resource_name is None:
    print(f"Creating new cluster: {CLUSTER_NAME}")
    cluster_resource_name = vertex_ray.create_ray_cluster(
        head_node_type=head_node,
        worker_node_types=worker_nodes,
        python_version=PYTHON_VERSION,
        ray_version=RAY_VERSION,
        cluster_name=CLUSTER_NAME,
        network=NETWORK,
        service_account=VERTEX_SERVICE_ACCOUNT,
    )

print("cluster_resource_name:", cluster_resource_name)


## Submit the training script as a Ray Job

`runtime_env.pip` ensures every worker has the libraries needed by `@ray.remote` tasks.
`working_dir="."` uploads the entrypoint script to the cluster.

In [ ]:
import shlex
import datetime
import google.auth
import google.auth.transport.requests
from ray.job_submission import JobSubmissionClient

dashboard_address = f"vertex_ray://{cluster_resource_name}"


def _get_ray_client():
    """Return a JobSubmissionClient with a fresh Google auth token."""
    credentials, _ = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    credentials.refresh(google.auth.transport.requests.Request())
    return JobSubmissionClient(
        dashboard_address,
        headers={"Authorization": f"Bearer {credentials.token}"},
    )


client = _get_ray_client()

ml_lags_str = ",".join(str(x) for x in ML_LAGS)
q = shlex.quote
run_ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
submission_id = f"nixtla-{SYS_ID_VALUE.lower().replace('_', '-')}-{run_ts}"

entrypoint = (
    f"python {q(ENTRYPOINT_SCRIPT.name)}"
    f" --project_id {q(PROJECT_ID)}"
    f" --bq_dataset {q(BQ_DATASET)}"
    f" --bq_table {q(BQ_TABLE)}"
    f" --uid_col {q(UID_COL)}"
    f" --date_col {q(DATE_COL)}"
    f" --target_col {q(TARGET_COL)}"
    f" --sys_id_value {q(SYS_ID_VALUE)}"
    f" --uid_delimiter {q(UID_DELIMITER)}"
    f" --freq {q(FREQ)}"
    f" --season_length {SEASON_LENGTH}"
    f" --horizon {HORIZON}"
    f" --validation_horizon {VALIDATION_HORIZON}"
    f" --min_history_for_full_audit {MIN_HISTORY_FOR_FULL_AUDIT}"
    f" --max_series {MAX_SERIES}"
    f" --ml_lags {q(ml_lags_str)}"
    f" --ml_audit_shards {ML_AUDIT_SHARDS}"
    f" --output_dataset {q(OUTPUT_DATASET)}"
    f" --forecast_table {q(FORECAST_TABLE)}"
    f" --metrics_table {q(METRICS_TABLE)}"
    f" --champions_table {q(CHAMPIONS_TABLE)}"
    f" --run_logs_table {q(RUN_LOGS_TABLE)}"
)

job_id = client.submit_job(
    entrypoint=entrypoint,
    submission_id=submission_id,
    runtime_env={
        "working_dir": str(ENTRYPOINT_DIR),
        "pip": [
            "immutabledict",
            "google-cloud-aiplatform",
            "google-cloud-bigquery",
            "pandas",
            "pyarrow",
            "db-dtypes",
            "statsforecast",
            "mlforecast",
            "utilsforecast",
            "lightgbm",
            "xgboost",
            "scikit-learn",
        ],
    },
)
print("job_id:", job_id)


## Tail logs until completion

In [ ]:
import time
from ray.job_submission import JobStatus

TERMINAL = {JobStatus.SUCCEEDED, JobStatus.FAILED, JobStatus.STOPPED}

last_logs = ""
token_refresh_interval = 60 * 45  # refresh token every 45 minutes
last_refresh = time.time()
client = _get_ray_client()

while True:
    # Refresh the client/token periodically to avoid 401s on long-running jobs
    if time.time() - last_refresh > token_refresh_interval:
        client = _get_ray_client()
        last_refresh = time.time()

    status = client.get_job_status(job_id)
    logs = client.get_job_logs(job_id)
    new = logs[len(last_logs):]
    if new:
        print(new, end="")
        last_logs = logs
    if status in TERMINAL:
        print(f"\n[done] status={status}")
        break
    time.sleep(5)


## Tear down the cluster

**Run this when you're done.** You're billed per-minute for all 5 VMs while the cluster exists.

In [ ]:
if DELETE_CLUSTER_AFTER_RUN:
    vertex_ray.delete_ray_cluster(cluster_resource_name)
    print("Cluster deleted.")
else:
    print(
        f"Skipping teardown (DELETE_CLUSTER_AFTER_RUN=False). Cluster left running: {cluster_resource_name}"
    )